# Hyperliquid consolidated account analytics

Combines balances, account history, fills, funding, positions, and PnL across the configured Hyperliquid wallets. Combined account-value history is shown only over the dates available for every wallet; charts show the oldest wallet's account history before shared coverage begins, then the combined account value.

Daily rollups and calendar-period cutoffs use `America/New_York`. Previously saved UTC-aggregated daily totals are mapped to Eastern date labels when loaded; their original intraday split cannot be reconstructed.


## Wallets and implementation

Edit `WALLET_ADDRESSES` here. Both addresses found in the single-wallet notebook are included: the active address and the previously commented address.


In [ ]:
from hl_multi_account_analytics_implementation import MultiWalletAnalytics

WALLET_ADDRESSES = [
    '0x...c248',
    '0x...07f2',
]

ATR_PERIOD = 14
ATR_STOP_MULTIPLE = 1.25
MARGIN_RESERVE_PCT = 20.0
AGGREGATE_RISK_PCT = 20.0

analytics=MultiWalletAnalytics(WALLET_ADDRESSES,data_dir='data')
analytics.position_atr_period=ATR_PERIOD
analytics.position_atr_stop_multiple=ATR_STOP_MULTIPLE
analytics.margin_reserve_pct=MARGIN_RESERVE_PCT
analytics.aggregate_risk_pct=AGGREGATE_RISK_PCT


## Fetch and consolidate wallet data

Fetch each wallet once and prepare combined account history and fills.


In [ ]:
%%capture
_ = analytics.fetch_data()


## Highlights

What went well and what needs attention since this notebook last ran, over the last 24 hours, and right now. PnL includes open positions and excludes deposits and withdrawals. Each run saves a baseline in `data/hl_highlights_state.json`.

In [ ]:
analytics.show_highlights()


## Account / PnL

Combined account and risk summary. 24H rows are rolling windows ending now and exclude deposits and withdrawals; daily statistics and calendar-period cutoffs use US Eastern days.


In [ ]:
analytics.show_account_pnl()


## Account charts

Account value and PnL charts. Before all wallets have saved history, account value and weekly returns use the oldest wallet alone; from shared coverage onward they use the combined accounts.


In [ ]:
analytics.show_account_charts()


## Account vs BTC

The daily account-vs-BTC comparison uses a time-weighted return across all wallets, so deposits, withdrawals, and transfers do not move it.


In [ ]:
analytics.show_account_vs_btc()


## Traded volume vs realized PnL chart

Daily fill volume and realized PnL pooled across wallets.


In [ ]:
analytics.show_volume_chart()


## Open positions

Show each wallet position separately, followed by an aggregate summary. The summary includes cumulative open-position funding as a percentage of combined account equity and total position value. Reserve-adjusted available margin is combined margin collateral (USDC on unified accounts) less the configured reserve; the summary shows that capacity, margin used, remaining available margin in dollars and as a percentage of that capacity, and position value as a percentage of equity. Set `MARGIN_RESERVE_PCT` (20% by default) in the setup cell to change the reserve. The position table includes funding and fees since open as dollar amounts, plus fees since open as a percentage of position value and position value as a percentage of wallet account value. Fee ratios are available for complete USDC-denominated fee history. Aggregate risk used and room apply `AGGREGATE_RISK_PCT` (20% of collateral by default) to `ATR_STOP_MULTIPLE` (1.25) x daily ATR over `ATR_PERIOD` (14 completed candles) plus an adverse five-day funding reserve; same-side positions across wallets are combined.

In [ ]:
analytics.show_open_positions()


## Net funding APR over time

Applies the open-positions `Net Funding APR (%) (simple, covered)` formula to Hyperliquid funding history: net funding cash flow divided by gross notional, annualized, pooled across wallets. The history records actual funding on positions held at the time; the summary table shows the current rate on current positions, so the two differ. Daily values use US Eastern days, and the 7-day line is notional-weighted. Positive means funding received.


In [ ]:
_ = analytics.show_funding_apr_chart()


## Realized PnL statistics

Pool spot/perp fills and signed funding across wallets.


In [ ]:
analytics.show_realized_pnl()


## Traded volume vs PnL

Combined turnover efficiency and correlations.


In [ ]:
analytics.show_volume_summary()


## PnL attribution

Combined realized and open-position PnL by asset.


In [ ]:
analytics.show_pnl_attribution()
